# Build a tiny RFdiffusion + ProteinMPNN in PyTorch

This notebook is an **architecture laboratory**. We will build small, inspectable models that preserve several important ideas from RFdiffusion and ProteinMPNN, then deliberately examine what has been left out.

By the end, we will have implemented:

1. a synthetic target–binder system;
2. Gaussian forward diffusion of binder coordinates;
3. a rotation/translation-equivariant coordinate denoiser;
4. a residue graph and message-passing sequence model;
5. random-order sequence decoding;
6. a transparent chemical interface score; and
7. reranking and direct score-guided sequence sampling.

### Interactive 3D controls

- **Rotate:** click and drag inside a 3D view.
- **Zoom:** use the mouse wheel or trackpad scroll.
- **Inspect:** hover over a residue marker for its index, class, and coordinates.
- **Animate:** use **Play/Pause** or drag the embedded frame slider in a diffusion view.
- **Reset:** double-click the view or use Plotly's home-camera button.

> **Scientific boundary:** these are teaching models, not reproductions of the trained production networks. The structures are procedural Cα traces, the alphabet contains six chemical classes, and the score is not a binding free energy or equilibrium constant.

## 0. The abstraction we are testing

```text
fixed target + noisy binder coordinates
                 │
                 ▼
       mini equivariant denoiser
                 │
                 ▼
          clean binder Cα trace
                 │
                 ▼
          mini sequence MPNN
                 │
                 ▼
       coarse chemical sequence
                 │
                 ▼
      interface score / guidance
```

RFdiffusion asks primarily **where residues should go**. ProteinMPNN asks **which amino acids should occupy those positions**. Our proposed chemical model acts most naturally on the second question, although bad backbone geometry can still make good chemistry impossible.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'affinity_benchmark').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import math
import inspect
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.rcParams.update({
    'text.usetex': False,  # keep plots independent of a system LaTeX install
    'font.family': 'DejaVu Sans',
    'font.serif': ['DejaVu Serif'],
})
from IPython.display import display, Markdown
import plotly.graph_objects as go

from affinity_benchmark.educational.mini_binder import (
    ALPHABET, TOKEN_NAMES, UNKNOWN_TOKEN,
    MiniEquivariantDenoiser, MiniProteinMPNN,
    apply_rotation, cosine_schedule, interface_score,
    make_synthetic_complex, q_sample, random_partial_sequence,
    random_rotation, sample_ddim, sample_with_score_guidance,
    tokens_to_string,
)

torch.manual_seed(7)
np.random.seed(7)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Repository:', ROOT)
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

def show_source(function_or_method):
    """Print the real source used by this notebook."""
    print(f'{function_or_method.__module__}.{function_or_method.__qualname__}')
    print(inspect.getsource(function_or_method))

Repository: /mnt/hdd2/BindingAffinityPredictor
PyTorch: 2.5.1+cu124
Device: cuda


## Function atlas: what every custom name does

Before following the pipeline, separate three kinds of Python object:

- A **function** takes inputs and returns outputs, for example `q_sample(x0, t, alpha_bar)`.
- A **class** is a blueprint containing trainable layers, for example `MiniEquivariantDenoiser(...)`. Calling the class constructs a model.
- A **method** belongs to an object, for example `mpnn.sample(...)` uses the already constructed and trained `mpnn`.

### Data and geometry helpers

| Name | Kind | What it does | Inputs → outputs | Learned? |
|---|---|---|---|---|
| `make_synthetic_complex` | function | Constructs procedural target and binder Cα helices, optionally rotates them, and assigns complementary toy chemistry labels. | lengths and batch size → coordinate and token tensors | No |
| `random_rotation` | function | Samples proper 3×3 rotation matrices using QR decomposition and corrects reflections so determinant = +1. | batch size → `[B,3,3]` | No; stochastic |
| `apply_rotation` | function | Applies one rotation matrix to every coordinate in each batch using `torch.einsum`. | `[B,L,3]`, `[B,3,3]` → `[B,L,3]` | No |
| `tokens_to_string` | function | Converts integer chemical-class tokens into readable strings such as `H++AAAAANNN`. | `[B,L]` → strings | No |

### Diffusion functions and model

| Name | Kind | What it does | Inputs → outputs | Learned? |
|---|---|---|---|---|
| `cosine_schedule` | function | Precomputes signal retention and noise variance at every timestep. | number of steps → three `[T]` arrays | No |
| `q_sample` | function | Implements forward diffusion and corrupts only the binder passed as `x0`. | clean binder, timesteps, schedule → noisy binder and sampled noise | No; stochastic unless noise supplied |
| `MiniEquivariantDenoiser` | class | Builds the trainable coordinate network using scalar messages and relative-vector coordinate updates. | architecture settings → model | Yes |
| `denoiser(xt, target, t)` | model call | Invokes `forward` and predicts a clean binder conditioned on a fixed target. | noisy binder, target, timesteps → `[B,Lb,3]` | Uses learned weights |
| `sample_ddim` | function | Repeatedly calls the trained denoiser to step toward timestep zero and saves every frame. | model, target, schedule, starting state → final binder and trajectory | Uses learned model |

### Sequence design, scoring, and guidance

| Name | Kind | What it does | Inputs → outputs | Learned? |
|---|---|---|---|---|
| `torch.cdist` | PyTorch function | Calculates every pairwise Euclidean distance for graph construction. | coordinates → distance matrix | No |
| `random_partial_sequence` | function | Reveals a random decoding prefix and marks remaining positions for prediction. | true tokens → partial tokens and Boolean mask | No; stochastic |
| `MiniProteinMPNN` | class | Builds the trainable invariant graph-message-passing sequence model. | architecture settings → model | Yes |
| `mpnn(...)` | model call | Returns six unnormalized class scores—logits—at every binder position. | structures and partial sequence → `[B,Lb,6]` | Uses learned weights |
| `mpnn.sample` | method | Repeatedly recomputes logits and samples one class at a time in an arbitrary order. | structure and temperature → sequences and log probabilities | Uses learned model; stochastic |
| `interface_score` | function | Sums smooth chemical contacts and subtracts clash and exposed-hydrophobe penalties. | complex coordinates and tokens → one scalar per candidate | No; hand-defined |
| `sample_with_score_guidance` | function | Adds `beta × chemical-score change` to MPNN logits before sampling. | trained MPNN, complex, beta → guided sequences | Hybrid |

### Visualization and training utilities

| Name | What it does |
|---|---|
| `_trace3d` | Converts one coordinate tensor into a Plotly line-and-marker trace with residue hover labels. |
| `interactive_complex` | Combines fixed-target and binder traces into a rotatable, zoomable 3D figure. |
| `interactive_trajectory` | Builds Plotly animation frames and play/pause/slider controls while replacing only the binder trace. |
| `F.mse_loss` | Averages squared coordinate differences between prediction and target. |
| `F.cross_entropy` | Measures how much probability the model assigns to the correct discrete class. |
| `optimizer.zero_grad` | Clears accumulated parameter gradients before a new backward pass. |
| `loss.backward` | Applies the chain rule through the recorded PyTorch computation graph. |
| `clip_grad_norm_` | Caps the total gradient norm to avoid unstable large updates. |
| `optimizer.step` | Uses Adam to update model parameters from their current gradients. |

The implementations are in [`mini_binder.py`](../src/affinity_benchmark/educational/mini_binder.py). We unpack the most important functions below rather than treating them as black boxes. At any point, run `show_source(q_sample)`, `show_source(interface_score)`, or `show_source(MiniProteinMPNN.forward)` to print the exact implementation.

---

## 1. Construct a system whose rule we know

Before training on real proteins, we use a procedural system with a known answer:

- the target and binder are idealized Cα helices;
- the target has repeating coarse chemical labels;
- each binder position is assigned the complement of its nearest target label.

The six tokens are:

| Token | Meaning | Preferred complement |
|---|---|---|
| `H` | hydrophobic | `H` |
| `D` | donor | `A` |
| `A` | acceptor | `D` |
| `+` | positive | `-` |
| `-` | negative | `+` |
| `N` | neutral | `N` |

Because we created the rule, we can distinguish **failure to learn** from ambiguity in experimental data. The price is that success here says nothing yet about real protein design.

In [ ]:
toy = make_synthetic_complex(batch_size=1, rotate=False)
print('target coordinates:', tuple(toy.target.shape))
print('binder coordinates:', tuple(toy.binder.shape))
print('target chemistry:   ', tokens_to_string(toy.target_tokens)[0])
print('binder complement:  ', tokens_to_string(toy.binder_tokens)[0])

def _trace3d(xyz, name, color, hover_prefix):
    xyz = torch.as_tensor(xyz).detach().cpu().numpy()
    hover = [f'{hover_prefix} residue {i+1}<br>x={x:.2f} Å<br>y={y:.2f} Å<br>z={z:.2f} Å'
             for i, (x, y, z) in enumerate(xyz)]
    return go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='lines+markers', name=name, text=hover, hoverinfo='text',
        line=dict(color=color, width=7), marker=dict(color=color, size=5),
    )

def interactive_complex(target, binder, title='', binder_color='#7656c3'):
    fig = go.Figure([
        _trace3d(target, 'fixed target', '#2b77ad', 'target'),
        _trace3d(binder, 'binder', binder_color, 'binder'),
    ])
    fig.update_layout(
        title=title, height=570, margin=dict(l=0, r=0, b=0, t=55),
        dragmode='orbit', uirevision='preserve-camera',
        scene=dict(
            aspectmode='data',
            xaxis_title='x (Å)', yaxis_title='y (Å)', zaxis_title='z (Å)',
        ),
        legend=dict(x=0.01, y=0.99),
    )
    fig.show(config={'scrollZoom': True, 'displaylogo': False})
    return fig

def interactive_trajectory(target, states, labels, title):
    # The fixed target is trace 0. Animation frames replace only trace 1,
    # which preserves target coordinates and the user's camera orientation.
    states = [torch.as_tensor(state).detach().cpu() for state in states]
    fig = go.Figure(
        data=[
            _trace3d(target, 'fixed target', '#2b77ad', 'target'),
            _trace3d(states[0], 'binder', '#7656c3', 'binder'),
        ],
        frames=[
            go.Frame(
                data=[_trace3d(state, 'binder', '#7656c3', 'binder')],
                traces=[1], name=str(index),
                layout=go.Layout(title_text=f'{title}<br><sup>{labels[index]}</sup>'),
            )
            for index, state in enumerate(states)
        ],
    )
    slider_steps = [dict(
        method='animate', label=str(index),
        args=[[str(index)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True},
                             'transition': {'duration': 0}}],
    ) for index in range(len(states))]
    fig.update_layout(
        title=f'{title}<br><sup>{labels[0]}</sup>', height=620,
        margin=dict(l=0, r=0, b=80, t=70), dragmode='orbit',
        uirevision='preserve-camera',
        scene=dict(aspectmode='data', xaxis_title='x (Å)',
                   yaxis_title='y (Å)', zaxis_title='z (Å)'),
        sliders=[dict(active=0, currentvalue=dict(prefix='frame: '),
                      pad=dict(t=45), steps=slider_steps)],
        updatemenus=[dict(
            type='buttons', direction='left', x=0, y=-0.08,
            buttons=[
                dict(label='▶ Play', method='animate',
                     args=[None, {'fromcurrent': True, 'frame': {'duration': 180, 'redraw': True},
                                  'transition': {'duration': 0}}]),
                dict(label='❚❚ Pause', method='animate',
                     args=[[None], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': False}}]),
            ],
        )],
    )
    fig.show(config={'scrollZoom': True, 'displaylogo': False})
    return fig

_ = interactive_complex(toy.target[0], toy.binder[0], 'Procedural Cα target–binder complex')

### Why begin with Cα coordinates?

A Cα-only representation makes diffusion and equivariance visible with very little code. But it cannot specify peptide-plane orientation, chirality, backbone hydrogen-bond geometry, or side-chain attachment direction.

Production RFdiffusion instead represents residue *i* by a translation and rotation:

$$X_i = (\mathbf r_i, \mathbf R_i), \qquad \mathbf r_i \in \mathbb R^3,\quad \mathbf R_i \in SO(3).$$

Here we first learn why translations need geometric treatment. A later notebook can add N–Cα–C frames and rotational diffusion.

## 2. Forward diffusion: deliberately destroy the binder

For a clean binder $X_0$, sample a noisy structure at timestep $t$:

$$X_t = \sqrt{\bar\alpha_t}X_0 + \sqrt{1-\bar\alpha_t}\epsilon, \qquad \epsilon\sim\mathcal N(0,I).$$

The target remains fixed. The model's learning problem is to recover $X_0$ from $X_t$, the target, and $t$. The diffusion index is a **noise level**, not molecular-dynamics time.

### `cosine_schedule(T)`

This produces three length-`T` arrays: `betas[t]` is incremental noise variance; `alphas[t] = 1 - betas[t]` is incremental signal retention; and `alpha_bar[t] = product(alphas[:t+1])` is cumulative signal retention. `q_sample` uses `alpha_bar` to jump directly from $X_0$ to any $X_t$ without simulating preceding steps.

### `q_sample(x0, timestep, alpha_bar, noise=None)` line by line

```python
def q_sample(x0, timestep, alpha_bar, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    a = alpha_bar[timestep].view(-1, 1, 1)
    xt = a.sqrt() * x0 + (1 - a).sqrt() * noise
    return xt, noise
```

For `x0` with shape `[B,Lb,3]`: (1) `randn_like` draws one standard-normal displacement per binder coordinate; (2) `alpha_bar[timestep]` selects one noise level per batch member; (3) `.view(-1,1,1)` changes `[B]` into `[B,1,1]`, allowing PyTorch to broadcast the scalar over all residues and Cartesian components; (4) `sqrt(a)*x0` is retained structure; and (5) `sqrt(1-a)*noise` is the added Gaussian component.

Both `xt` and the actual `noise` are returned because diffusion models can be trained to predict either $X_0$ or $\epsilon$. Our model predicts $X_0$. The target is not an argument, so `q_sample` cannot change it: only the binder tensor supplied as `x0` is diffused. Passing `noise=fixed_noise` makes every animation frame use the same underlying random draw.

In [ ]:
show_source(q_sample)

T = 30
schedule_cpu = cosine_schedule(T)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(T), schedule_cpu['alpha_bar'], color='#2c745b', lw=3)
ax.set(xlabel='diffusion timestep t', ylabel=r'$\bar\alpha_t$',
       title='Signal retained under the cosine schedule')
ax.grid(alpha=.2);
plt.show()

In [ ]:
fixed_noise = torch.randn_like(toy.binder)
forward_states, forward_labels = [], []
for t in range(T):
    timestep = torch.tensor([t])
    xt, _ = q_sample(toy.binder, timestep, schedule_cpu['alpha_bar'], fixed_noise)
    retained = schedule_cpu['alpha_bar'][t].item()
    forward_states.append(xt[0])
    forward_labels.append(f't={t}; signal retained={retained:.3f}')

_ = interactive_trajectory(
    toy.target[0], forward_states, forward_labels, 'Forward diffusion'
)

## 3. Why an equivariant denoiser?

Protein coordinates have no privileged laboratory orientation. If we rotate the entire input by $R$ and translate it by $b$, a physically consistent coordinate model should obey

$$f(RX+b)=Rf(X)+b.$$

Our message block calculates scalar messages from invariant distances, then uses those scalars to weight relative vectors:

$$m_{ij}=\phi_m(h_i,h_j,\lVert x_i-x_j\rVert),$$

$$x'_i=x_i+\sum_j(x_i-x_j)\phi_x(m_{ij}).$$

Distances do not change under rotation or translation. Relative vectors rotate but do not translate. The complete update is therefore rotation- and translation-equivariant by construction.

In [ ]:
denoiser = MiniEquivariantDenoiser(hidden_dim=64, layers=4, max_steps=T).to(DEVICE)
parameter_count = sum(p.numel() for p in denoiser.parameters())
print(f'Mini denoiser parameters: {parameter_count:,}')

probe = make_synthetic_complex(batch_size=2, rotate=False, device=DEVICE)
probe_t = torch.tensor([5, 17], device=DEVICE)
probe_xt = probe.binder + 0.4 * torch.randn_like(probe.binder)
with torch.no_grad():
    original_output = denoiser(probe_xt, probe.target, probe_t)

rotation = random_rotation(2, device=DEVICE)
translation = torch.tensor([[[2., -3., 1.]], [[-1., 4., .5]]], device=DEVICE)
rotated_xt = apply_rotation(probe_xt, rotation) + translation
rotated_target = apply_rotation(probe.target, rotation) + translation
with torch.no_grad():
    transformed_output = denoiser(rotated_xt, rotated_target, probe_t)
expected_output = apply_rotation(original_output, rotation) + translation
equivariance_error = (transformed_output - expected_output).abs().max().item()
print(f'Maximum equivariance error: {equivariance_error:.3e} Å')
print('This property holds before training because it is built into the operations.')

### Train the denoiser

Every training example is randomly rotated. We sample one timestep per example, corrupt the binder, and ask the network to predict the clean coordinates.

The main loss is coordinate mean-squared error. A small adjacent-Cα distance penalty illustrates how geometric priors can stabilize a tiny model:

$$\mathcal L = \lVert\hat X_0-X_0\rVert^2 + \lambda\sum_i(\lVert x_{i+1}-x_i\rVert-3.8\,\text{Å})^2.$$

Production RFdiffusion was initialized from a much larger RoseTTAFold structure-prediction network and trained on real protein structures. Our tiny network has neither advantage.

In [ ]:
schedule = {key: value.to(DEVICE) for key, value in schedule_cpu.items()}
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
DENOISER_STEPS = 450
denoiser_history = []
denoiser.train()

for step in range(DENOISER_STEPS):
    # 1) Create a fresh batch and randomly rotate every complete complex.
    batch = make_synthetic_complex(
        batch_size=64, rotate=True, coordinate_noise=0.03, device=DEVICE
    )
    # 2) Give each example its own randomly selected diffusion noise level.
    t = torch.randint(0, T, (64,), device=DEVICE)
    # 3) Corrupt binder coordinates; batch.target remains unchanged.
    xt, _ = q_sample(batch.binder, t, schedule['alpha_bar'])
    # 4) Predict clean binder coordinates conditioned on the fixed target.
    predicted_x0 = denoiser(xt, batch.target, t)

    # 5) Compare prediction with truth and mildly regularize Cα spacing.
    coordinate_loss = F.mse_loss(predicted_x0, batch.binder)
    bond_lengths = (predicted_x0[:, 1:] - predicted_x0[:, :-1]).norm(dim=-1)
    bond_loss = ((bond_lengths - 3.8) ** 2).mean()
    loss = coordinate_loss + 0.05 * bond_loss

    # 6) Remove gradients left from the preceding optimization step.
    optimizer.zero_grad()
    # 7) Backpropagate parameter derivatives through the computation graph.
    loss.backward()
    # 8) Limit rare, very large gradients for numerical stability.
    torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
    # 9) Adam changes the trainable weights using those gradients.
    optimizer.step()
    denoiser_history.append(loss.item())

    if step % 100 == 0 or step == DENOISER_STEPS - 1:
        print(f'step {step:4d} | loss {loss.item():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(denoiser_history, color='#7656c3', alpha=.35)
window = 25
smoothed = np.convolve(denoiser_history, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(denoiser_history)), smoothed, color='#4f347f', lw=2)
ax.set(xlabel='optimization step', ylabel='training loss', title='Mini denoiser training')
ax.set_yscale('log'); ax.grid(alpha=.2);
plt.show()

### Watch a reverse trajectory

We begin with **partial diffusion**: a known synthetic binder is corrupted to an intermediate timestep and reconstructed. This is easier than generating a protein from nearly pure noise and gives an honest first check of the denoising machinery.

A production-quality unconditional generator requires far more data, model capacity, training, and a full residue-frame representation.

In [ ]:
evaluation = make_synthetic_complex(batch_size=1, rotate=False, device=DEVICE)
START_T = 20
start_tensor = torch.tensor([START_T], device=DEVICE)
starting_state, _ = q_sample(evaluation.binder, start_tensor, schedule['alpha_bar'])
final_binder, reverse_trajectory = sample_ddim(
    denoiser, evaluation.target, evaluation.binder.shape[1], schedule,
    initial_noise=starting_state, start_step=START_T,
)
reconstruction_mse = F.mse_loss(final_binder, evaluation.binder).item()
print(f'Partial-diffusion reconstruction MSE: {reconstruction_mse:.3f} Å²')

reverse_states = [state[0] for state in reverse_trajectory]
reverse_labels = [
    f'reverse frame {frame}; remaining noise level t≈{max(START_T-frame, 0)}'
    for frame in range(len(reverse_states))
]
_ = interactive_trajectory(
    evaluation.target[0].cpu(), reverse_states, reverse_labels,
    'Partial reverse diffusion',
)

### What did we preserve, and what did we simplify?

| Feature | This notebook | RFdiffusion | Why it matters |
|---|---|---|---|
| Geometric variables | Cα coordinates | Cα translation + N–Cα–C orientation | Frames encode local backbone direction |
| Forward process | Gaussian translation noise | translation and rotational diffusion | Proteins require both position and orientation |
| Denoiser | small equivariant message-passing network | RoseTTAFold-derived three-track network | Pretraining supplies a strong protein-structure prior |
| Data | procedural helices | protein structures | Real folds and interfaces are vastly more diverse |
| Conditioning | fixed target and residue index | motifs, contigs, sequence, hotspots, topology and coordinates | Enables diverse design tasks |
| Sampling shown first | partial denoising | partial or full generation | Full generation is the more difficult test |

## 4. Mini-ProteinMPNN: structure becomes a residue graph

After obtaining a backbone, sequence design becomes a graph problem:

- every residue is a node;
- spatially neighboring residues exchange messages;
- edge features encode distance and whether two residues belong to the same chain;
- known target chemistry and already decoded binder tokens enter as node features.

The model returns one logit for every chemical class at every binder position. Unlike a language model, there is no privileged N-to-C decoding direction. We therefore train with random partial decoding contexts and sample in arbitrary residue orders.

### `torch.cdist` and graph construction

`torch.cdist(coordinates, coordinates)` calculates $d_{ij}=\lVert x_i-x_j\rVert_2$ for every residue pair. Sorting each row and discarding its first entry—the residue's zero-distance connection to itself—gives nearest spatial neighbors. These edges can connect residues far apart in sequence or on different chains.

### `MiniProteinMPNN(...)` versus `mpnn(...)`

`MiniProteinMPNN(hidden_dim=64, layers=3)` constructs layers and initializes parameters; it does not train them. Later, `mpnn(binder, target, target_tokens, partial)` invokes the model's `forward` method and returns logits of shape `[batch,binder_length,6]`. A logit is an unnormalized score; `softmax(logits)` converts the six scores into probabilities.

In [ ]:
graph_data = make_synthetic_complex(batch_size=1, rotate=False)
coordinates = torch.cat((graph_data.binder[0], graph_data.target[0]), dim=0)
distances = torch.cdist(coordinates, coordinates)
k = 4
neighbors = distances.argsort(dim=-1)[:, 1:k+1]

edge_x, edge_y, edge_z = [], [], []
for i in range(len(coordinates)):
    for j in neighbors[i]:
        pair = coordinates[[i, int(j)]].cpu().numpy()
        edge_x.extend([pair[0, 0], pair[1, 0], None])
        edge_y.extend([pair[0, 1], pair[1, 1], None])
        edge_z.extend([pair[0, 2], pair[1, 2], None])
lb = graph_data.binder.shape[1]
binder_xyz = coordinates[:lb].cpu().numpy()
target_xyz = coordinates[lb:].cpu().numpy()
binder_text = [f'binder {i+1}<br>class={ALPHABET[token]}'
               for i, token in enumerate(graph_data.binder_tokens[0].tolist())]
target_text = [f'target {i+1}<br>class={ALPHABET[token]}'
               for i, token in enumerate(graph_data.target_tokens[0].tolist())]
graph_figure = go.Figure([
    go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines',
                 line=dict(color='#aeb9b5', width=2), opacity=.35,
                 hoverinfo='skip', name=f'{k}-NN edges'),
    go.Scatter3d(x=binder_xyz[:,0], y=binder_xyz[:,1], z=binder_xyz[:,2],
                 mode='markers', marker=dict(color='#7656c3', size=7),
                 text=binder_text, hoverinfo='text', name='binder nodes'),
    go.Scatter3d(x=target_xyz[:,0], y=target_xyz[:,1], z=target_xyz[:,2],
                 mode='markers', marker=dict(color='#2b77ad', size=7),
                 text=target_text, hoverinfo='text', name='target nodes'),
])
graph_figure.update_layout(
    title=f'Residue graph: {k} nearest neighbors per node', height=570,
    margin=dict(l=0,r=0,b=0,t=55), dragmode='orbit',
    scene=dict(aspectmode='data', xaxis_title='x (Å)',
               yaxis_title='y (Å)', zaxis_title='z (Å)'),
)
graph_figure.show(config={'scrollZoom': True, 'displaylogo': False})

In [ ]:
mpnn = MiniProteinMPNN(hidden_dim=64, layers=3).to(DEVICE)
print(f'Mini sequence-model parameters: {sum(p.numel() for p in mpnn.parameters()):,}')

demo = make_synthetic_complex(batch_size=2, device=DEVICE)
partial, predict_mask = random_partial_sequence(demo.binder_tokens)
logits = mpnn(demo.binder, demo.target, demo.target_tokens, partial)
print('partial tokens:', partial[0].tolist(), f'(unknown token = {UNKNOWN_TOKEN})')
print('logit tensor:  ', tuple(logits.shape), '= batch × binder positions × classes')

### Train with random decoding contexts

For each example we choose a random residue order, reveal a random prefix, and calculate cross-entropy on the unrevealed positions. This compact masked decoder is **ProteinMPNN-like**, not a line-for-line reproduction of ProteinMPNN's causal masking implementation.

The reason for random order is preserved: any structurally important position can be decoded early, and later choices can condition on it.

`random_partial_sequence(labels)` returns `partial`, containing true tokens at randomly selected already-decoded positions and `UNKNOWN_TOKEN` elsewhere, plus a Boolean `predict_mask`. `True` marks positions included in the loss.

`F.cross_entropy(logits[predict_mask], labels[predict_mask])` applies log-softmax, selects the log probability assigned to the correct class, negates it, and averages across predicted positions. Minimizing it increases probability assigned to the known procedural answer.

In [ ]:
mpnn_optimizer = torch.optim.Adam(mpnn.parameters(), lr=2e-3)
MPNN_STEPS = 250
mpnn_history = []
mpnn.train()

for step in range(MPNN_STEPS):
    batch = make_synthetic_complex(
        batch_size=64, rotate=True, coordinate_noise=0.08, device=DEVICE
    )
    partial, predict_mask = random_partial_sequence(batch.binder_tokens)
    logits = mpnn(batch.binder, batch.target, batch.target_tokens, partial)
    loss = F.cross_entropy(logits[predict_mask], batch.binder_tokens[predict_mask])

    mpnn_optimizer.zero_grad()
    loss.backward()
    mpnn_optimizer.step()
    mpnn_history.append(loss.item())

    if step % 50 == 0 or step == MPNN_STEPS - 1:
        print(f'step {step:3d} | cross-entropy {loss.item():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].plot(mpnn_history, color='#2c745b')
axes[0].set(xlabel='step', ylabel='cross-entropy', title='Sequence-model training')
axes[0].set_yscale('log'); axes[0].grid(alpha=.2)

test = make_synthetic_complex(batch_size=64, device=DEVICE)
sampled, logp = mpnn.sample(test.binder, test.target, test.target_tokens, temperature=0.7)
accuracy = (sampled == test.binder_tokens).float().mean().item()
counts = torch.bincount(sampled.flatten(), minlength=len(ALPHABET)).cpu()
axes[1].bar(ALPHABET, counts, color='#7656c3')
axes[1].set(xlabel='coarse class', ylabel='sampled count', title=f'Samples; rule recovery={accuracy:.1%}')
plt.tight_layout(); plt.show()
print('example prediction:', tokens_to_string(sampled[:1])[0])
print('known complement:  ', tokens_to_string(test.binder_tokens[:1])[0])

## 5. Add an explicit interface-quality score

For binder class $s_i$ and target class $t_j$, define a smooth contact

$$c_{ij}=\sigma\left(\frac{r_c-d_{ij}}{w}\right),$$

and a transparent compatibility table $C(s_i,t_j)$. The toy score is

$$F=\sum_{ij}c_{ij}C(s_i,t_j)-2E_{\mathrm{clash}}-0.5E_{\mathrm{exposed\ hydrophobe}}.$$

This deliberately omits solvent ensembles, entropy, protonation, unbound-state stability, and target flexibility. We therefore call it an **interface-quality score**, never $\Delta G_{bind}$.

### `interface_score(...)`

The function performs four deterministic operations: (1) `torch.cdist` constructs all binder–target Cα distances; (2) a sigmoid turns distance into a smooth contact weight; (3) a fixed 6×6 compatibility matrix supplies chemical rewards and penalties; and (4) steric-overlap and exposed-hydrophobe penalties are subtracted.

It returns one scalar per candidate, shape `[B]`. No parameters are fitted and no gradient update occurs. Higher is defined as better only under this toy convention.

In [ ]:
candidate_batch = make_synthetic_complex(batch_size=80, rotate=False, device=DEVICE)
torch.manual_seed(11)
candidates, mpnn_logp = mpnn.sample(
    candidate_batch.binder, candidate_batch.target, candidate_batch.target_tokens,
    temperature=1.4,
)
scores = interface_score(
    candidate_batch.binder, candidate_batch.target, candidates, candidate_batch.target_tokens
).cpu()
logp_cpu = mpnn_logp.cpu()
best = scores.argmax().item()
worst = scores.argmin().item()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(scores.numpy(), bins=18, color='#2c745b', alpha=.8)
axes[0].axvline(scores[best], color='#17221f', ls='--', label='reranked best')
axes[0].set(xlabel='interface-quality score ↑', ylabel='candidate count', title='Generate, then rerank')
axes[0].legend()
axes[1].scatter(logp_cpu, scores, c='#7656c3', alpha=.65)
axes[1].set(xlabel='ProteinMPNN log probability ↑', ylabel='interface score ↑',
            title='Related, but distinct objectives')
plt.tight_layout(); plt.show()

print(f'best  : {tokens_to_string(candidates[best:best+1])[0]} | score={scores[best]:.2f} | logP={logp_cpu[best]:.2f}')
print(f'worst : {tokens_to_string(candidates[worst:worst+1])[0]} | score={scores[worst]:.2f} | logP={logp_cpu[worst]:.2f}')

## 6. Put the score inside sequence decoding

At sequence position $i$, the MPNN emits logits $\ell_i(a)$. For every possible class $a$, we calculate the change in interface score and adjust the logits:

$$\ell_i^{guided}(a)=\ell_i^{MPNN}(a)+\beta\,\Delta F_i(a).$$

- $\beta=0$ gives ordinary MPNN sampling.
- moderate $\beta$ biases the existing sequence distribution.
- very large $\beta$ lets the score dominate and can destroy diversity or exploit score defects.

The helper below provisionally treats not-yet-decoded positions as neutral. That is an explicit approximation; a richer model could integrate over uncertain future identities or optimize the complete sequence.

### `sample_with_score_guidance(...)`

At each position this function: (1) asks the MPNN for six logits; (2) substitutes each possible class at the current position; (3) evaluates six temporary sequences with `interface_score`; (4) mean-centers those scores; (5) adds `beta × score` to the original logits; and (6) samples from the adjusted categorical distribution.

It returns the guided sequence and its log probability under the **original**, unguided MPNN. That second output measures how far chemical guidance pushes sampling away from the learned sequence distribution.

In [ ]:
betas = [0.0, 5.0, 20.0]
guided_results = {}
for beta in betas:
    comparison = make_synthetic_complex(batch_size=24, rotate=False, device=DEVICE)
    fixed_order = torch.arange(comparison.binder.shape[1], device=DEVICE).repeat(24, 1)
    torch.manual_seed(21)
    sequence, original_logp = sample_with_score_guidance(
        mpnn, comparison.binder, comparison.target, comparison.target_tokens,
        beta=beta, temperature=1.4, order=fixed_order,
    )
    score = interface_score(
        comparison.binder, comparison.target, sequence, comparison.target_tokens
    )
    guided_results[beta] = (sequence.cpu(), original_logp.cpu(), score.cpu())

means = [guided_results[b][2].mean().item() for b in betas]
diversity = [len(set(tokens_to_string(guided_results[b][0]))) for b in betas]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar([str(b) for b in betas], means, color=['#2b77ad','#7656c3','#2c745b'])
axes[0].set(xlabel='guidance β', ylabel='mean interface score ↑', title='Score adherence')
axes[1].bar([str(b) for b in betas], diversity, color=['#2b77ad','#7656c3','#2c745b'])
axes[1].set(xlabel='guidance β', ylabel='unique sequences / 24', title='Sequence diversity')
plt.tight_layout(); plt.show()
for beta in betas:
    seq, lp, score = guided_results[beta]
    print(f'beta={beta:>3}: mean score={score.mean():6.2f}, mean original logP={lp.mean():6.2f}, unique={len(set(tokens_to_string(seq))):2d}')

## 7. What this exercise establishes

We have now made several architectural choices concrete:

1. **Diffusion provides a controlled denoising curriculum.** The timestep tells the model how corrupted the coordinates are.
2. **Equivariance prevents the network from relearning arbitrary coordinate-system symmetries.** We verified it numerically.
3. **Residue graphs follow physical proximity, not only sequence adjacency.** This is essential at folded cores and binding interfaces.
4. **Random decoding order fits protein geometry better than a privileged left-to-right convention.**
5. **The MPNN likelihood and chemical score answer different questions.** Combining them retains a learned structural prior while adding an explicit interface objective.
6. **Guidance strength is a scientific hyperparameter.** It controls score adherence, likelihood, and diversity and must be selected before evaluating outcomes.

What we have *not* established is equally important: this toy model has not learned real protein structure, real amino-acid energetics, experimental binding, specificity, or thermodynamic free energy.

## 8. Suggested next lessons

A scientifically useful progression would be:

1. **Inspect every tensor in one message-passing block.** Draw the scalar messages and coordinate vectors.
2. **Ablate equivariance.** Replace relative geometry with raw coordinates and measure rotation generalization.
3. **Add N–Cα–C residue frames.** Introduce rotations in $SO(3)$ and compare with Cα-only ambiguity.
4. **Use all 20 amino acids.** Separate residue identity from rotameric side-chain geometry.
5. **Add complete-sequence Monte Carlo.** Demonstrate coupled mutations and score frustration.
6. **Move to a small, versioned real-structure dataset.** Define homology-aware train/validation/test splits before download.
7. **Compare against the real pretrained models.** Run the miniature and production pipelines on identical small examples without confusing their output meanings.

### Primary sources

- Watson et al., [De novo design of protein structure and function with RFdiffusion](https://doi.org/10.1038/s41586-023-06415-8), *Nature* (2023).
- Dauparas et al., [Robust deep learning–based protein sequence design using ProteinMPNN](https://doi.org/10.1126/science.add2187), *Science* (2022).
- Baek et al., [Accurate prediction of protein structures and interactions using a three-track neural network](https://doi.org/10.1126/science.abj8754), *Science* (2021).